In [1]:
from itertools import islice
from time import perf_counter

from hasil_ta_notebook import all_latin_squares, build_permutation_context, is_commutative_pair


KNOWN_TOTALS = {
    3: 12,
    4: 576,
    5: 161280,
}


def sigma_max_index(A, ctx):
    n = len(A)
    sigma = tuple(row.index(n) for row in A)
    return ctx["perm_index"][sigma]


def benchmark_exact(n):
    ctx = build_permutation_context(n)
    commute = ctx["commute"]
    latin_squares = list(all_latin_squares(n))
    sigma_indices = [sigma_max_index(A, ctx) for A in latin_squares]

    started = perf_counter()
    total_pairs = 0
    commutative_pairs = 0
    for A in latin_squares:
        for B in latin_squares:
            total_pairs += 1
            if is_commutative_pair(A, B):
                commutative_pairs += 1
    brute_force_time = perf_counter() - started

    started = perf_counter()
    theorem_candidates = 0
    theorem_commutative_pairs = 0
    for i, A in enumerate(latin_squares):
        sigma_a = sigma_indices[i]
        for j, B in enumerate(latin_squares):
            if commute[sigma_a][sigma_indices[j]]:
                theorem_candidates += 1
                if is_commutative_pair(A, B):
                    theorem_commutative_pairs += 1
    theorem_time = perf_counter() - started

    return {
        "orde": n,
        "jumlah_latin_square": len(latin_squares),
        "total_pasangan": total_pairs,
        "kandidat_teorema": theorem_candidates,
        "pasangan_komutatif": commutative_pairs,
        "pasangan_komutatif_teorema": theorem_commutative_pairs,
        "waktu_bruteforce": brute_force_time,
        "waktu_teorema": theorem_time,
        "faktor_percepatan": brute_force_time / theorem_time if theorem_time else None,
    }


def estimate_order_five(sample_repetitions=20000):
    n = 5
    ctx = build_permutation_context(n)
    commute = ctx["commute"]
    latin_sample = list(islice(all_latin_squares(n), 2))
    A = latin_sample[0]
    B = latin_sample[1]
    sigma_a = sigma_max_index(A, ctx)
    sigma_b = sigma_max_index(B, ctx)

    started = perf_counter()
    for _ in range(sample_repetitions):
        is_commutative_pair(A, B)
    pair_time = (perf_counter() - started) / sample_repetitions

    started = perf_counter()
    for _ in range(sample_repetitions):
        commute[sigma_a][sigma_b]
    filter_time = (perf_counter() - started) / sample_repetitions

    total_pairs = KNOWN_TOTALS[n] ** 2
    theorem_ratio = 7 / 120
    theorem_candidates = int(total_pairs * theorem_ratio)
    brute_force_time = total_pairs * pair_time
    theorem_time = total_pairs * filter_time + theorem_candidates * pair_time

    return {
        "orde": n,
        "jumlah_latin_square": KNOWN_TOTALS[n],
        "total_pasangan": total_pairs,
        "kandidat_teorema": theorem_candidates,
        "waktu_bruteforce": brute_force_time,
        "waktu_teorema": theorem_time,
        "faktor_percepatan": brute_force_time / theorem_time if theorem_time else None,
        "rata_waktu_uji_pasangan": pair_time,
        "rata_waktu_filter": filter_time,
    }


def format_duration(seconds):
    if seconds < 60:
        return f"{seconds:.3f} detik"
    if seconds < 3600:
        return f"{seconds / 60:.2f} menit"
    if seconds < 86400:
        return f"{seconds / 3600:.2f} jam"
    return f"{seconds / 86400:.2f} hari"


A ke-1: 1 pasangan, valid=True
A ke-2: 6 pasangan, valid=True
A ke-3: 6 pasangan, valid=True
A ke-4: 1 pasangan, valid=True
A ke-5: 1 pasangan, valid=True
A ke-6: 6 pasangan, valid=True
A ke-7: 6 pasangan, valid=True
A ke-8: 1 pasangan, valid=True
A ke-9: 1 pasangan, valid=True
A ke-10: 6 pasangan, valid=True
A ke-11: 6 pasangan, valid=True
A ke-12: 1 pasangan, valid=True
A ke-1: 48 pasangan, valid=True
A ke-2: 42 pasangan, valid=True
A ke-3: 24 pasangan, valid=True
A ke-4: 22 pasangan, valid=True
A ke-5: 28 pasangan, valid=True
A ke-6: 5 pasangan, valid=True
A ke-7: 12 pasangan, valid=True
A ke-8: 1 pasangan, valid=True
A ke-9: 40 pasangan, valid=True
A ke-10: 3 pasangan, valid=True
A ke-11: 16 pasangan, valid=True
A ke-12: 1 pasangan, valid=True
A ke-13: 1 pasangan, valid=True
A ke-14: 4 pasangan, valid=True
A ke-15: 16 pasangan, valid=True
A ke-16: 4 pasangan, valid=True
A ke-17: 1 pasangan, valid=True
A ke-18: 32 pasangan, valid=True
A ke-19: 4 pasangan, valid=True
A ke-20: 18 pasa

In [2]:
hasil_orde_3 = benchmark_exact(3)
hasil_orde_3


{'orde': 3,
 'jumlah_latin_square': 12,
 'total_pasangan': 144,
 'kandidat_teorema': 72,
 'pasangan_komutatif': 42,
 'pasangan_komutatif_teorema': 42,
 'waktu_bruteforce': 0.004007199953775853,
 'waktu_teorema': 0.0019014000426977873,
 'faktor_percepatan': 2.107499665399327}

In [3]:
hasil_orde_4 = benchmark_exact(4)
hasil_orde_4


{'orde': 4,
 'jumlah_latin_square': 576,
 'total_pasangan': 331776,
 'kandidat_teorema': 69120,
 'pasangan_komutatif': 10080,
 'pasangan_komutatif_teorema': 10080,
 'waktu_bruteforce': 8.192869199963752,
 'waktu_teorema': 1.8069121000007726,
 'faktor_percepatan': 4.534182487327551}

In [4]:
estimasi_orde_5 = estimate_order_five()
estimasi_orde_5


{'orde': 5,
 'jumlah_latin_square': 161280,
 'total_pasangan': 26011238400,
 'kandidat_teorema': 1517322240,
 'waktu_bruteforce': 1060476.4988523673,
 'waktu_teorema': 62434.02659705115,
 'faktor_percepatan': 16.985553497881476,
 'rata_waktu_uji_pasangan': 4.076993500057142e-05,
 'rata_waktu_filter': 2.2024998906999826e-08}

In [5]:
for hasil in [hasil_orde_3, hasil_orde_4, estimasi_orde_5]:
    label = f"orde {hasil['orde']}"
    if hasil['orde'] == 5:
        label += " (estimasi)"
    print(
        f"{label}: tanpa teorema {format_duration(hasil['waktu_bruteforce'])}, "
        f"dengan teorema {format_duration(hasil['waktu_teorema'])}, "
        f"percepatan {hasil['faktor_percepatan']:.2f}x"
    )


orde 3: tanpa teorema 0.004 detik, dengan teorema 0.002 detik, percepatan 2.11x
orde 4: tanpa teorema 8.193 detik, dengan teorema 1.807 detik, percepatan 4.53x
orde 5 (estimasi): tanpa teorema 12.27 hari, dengan teorema 17.34 jam, percepatan 16.99x
